# Figure 1 — analysis (human dermal fibroblasts, young 10 yo vs old 75 yo)
Computes every Figure 1 / Supplementary Figure 1S result and writes it to `data/`; the panels are drawn from these tables
in `fibroblast_figures.ipynb` and the pages assembled in `fibroblast_pages.ipynb`.

Run `fibroblast_cohort.ipynb` (population cohort) first: its per-donor interferon-γ scores are
read here. Primary Hi-C and RNA files stay under `/data/land/3d-inter-paper-data/`.

In [1]:
import json
import os
import pickle
import sys
from collections import defaultdict

import cooler
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

GV = "../.."                                            # github-version root
sys.path.insert(0, f"{GV}/notebooks")                   # shared statistics modules
import coloc_stats                                      # per-state hypergeometric test on gene-gene contacts
import gsea                                             # preranked GSEA on the DESeq2 Wald statistic
import kcore                                            # state-specific k-cores
import locus_null                                       # locus-matched permutation test

META = f"{GV}/metadata"
DATA = "data"
COOLERS = "/data/land/3d-inter-paper-data/normalized_hic_cooler"
G_ALL = "/data/land/3d-inter-paper-data/networkx_graphs/G_all.pkl"
BULK = "/data/land/3d-inter-paper-data/bulk-rna-data/RNAseq/old_vs_young_bulk.csv"
DONOR_DESEQ = "/data/land/3d-inter-paper-data/bulk-rna-data/old_vs_young_deseq_annotated.csv"
GMT = f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.gmt"    # the 60-pathway panel as gene sets (read only)
SIG = 0.05                                              # FDR threshold
pathways = json.load(open(f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.json"))   # the 60-pathway panel

## Hi-C processing (Figure 1a)
Interchromosomal (KR-normalized) Hi-C and LAS-processed binary contacts for one replicate per age (B1R1), and the
age-specific map (young-only / old-only / shared LAS contacts), pooled into 16 × 16-bin (4 Mb) blocks for display.
Writes `data/fig1a_pooled_maps.npz`.

In [2]:
F = 16              # bins pooled per displayed pixel (16 x 250 kb = 4 Mb)
CHUNK = 5_000_000   # pixels read per chunk
MAPS = {"young_raw": ("YoungB1R1_normalized_BP250000/YoungB1R1_from_INTERKR_post_processed.250000.cool", "mean"),
        "old_raw": ("OldB1R1_normalized_BP250000/OldB1R1_from_INTERKR_post_processed.250000.cool", "mean"),
        "young_las": ("YoungB1R1_normalized_BP250000/YoungB1R1_from_INTERKR_post_processed_LAS_only_binary.250000.cool", "any"),
        "old_las": ("OldB1R1_normalized_BP250000/OldB1R1_from_INTERKR_post_processed_LAS_only_binary.250000.cool", "any"),
        "age_specific": ("diff_mtx_LAS_only_binary.250000.cool", "state")}


def pooled_matrix(path, mode):
    """Interchromosomal matrix pooled into F x F bin blocks, built from the sparse pixels (no dense 11k x 11k matrix).

    mode "mean": mean contact frequency per block; "any": 1 if the block has any contact;
    "state": age-specific codes (1 young-only, 2 old-only, 3 shared) -> the most common code in the block (0 = none).
    Returns (pooled matrix, start of each chromosome in pooled units).
    """
    clr = cooler.Cooler(path)
    chrom = clr.bins()["chrom"][:].astype(str).values
    n = -(-len(chrom) // F)                               # ceil division: pooled matrix size
    layers = 4 if mode == "state" else 1
    acc = np.zeros(layers * n * n)
    for lo in range(0, clr.info["nnz"], CHUNK):
        px = clr.pixels()[lo:lo + CHUNK]
        i, j, v = px.bin1_id.values, px.bin2_id.values, px["count"].values
        inter = chrom[i] != chrom[j]                      # interchromosomal pixels only
        i, j, v = i[inter] // F, j[inter] // F, v[inter]
        layer = v.astype(int) if mode == "state" else np.zeros(len(v), int)
        w = v if mode == "mean" else np.ones(len(v))
        for a, b in ((i, j), (j, i)):                     # cooler stores the upper triangle; mirror it
            acc += np.bincount((layer * n + a) * n + b, weights=w, minlength=layers * n * n)
    acc = acc.reshape(layers, n, n)
    if mode == "mean":
        mat = acc[0] / F ** 2
    elif mode == "any":
        mat = (acc[0] > 0).astype(int)
    else:
        mat = np.where(acc[1:].sum(0) > 0, acc[1:].argmax(0) + 1, 0)
    starts = [clr.extent(c)[0] / F for c in clr.chromnames]
    return mat, starts

In [3]:
mats = {}
for key, (rel, mode) in MAPS.items():
    mats[key], chrom_starts = pooled_matrix(f"{COOLERS}/{rel}", mode)
    print(key, mats[key].shape)
np.savez_compressed(f"{DATA}/fig1a_pooled_maps.npz", chrom_starts=np.array(chrom_starts), **mats)

codes = np.unique(mats["age_specific"], return_counts=True)
print("age-specific blocks (0 none, 1 young-only, 2 old-only, 3 shared):", dict(zip(*codes)))

young_raw (720, 720)


old_raw (720, 720)
young_las (720, 720)


old_las (720, 720)


age_specific (720, 720)


age-specific blocks (0 none, 1 young-only, 2 old-only, 3 shared): {0: 495009, 1: 1694, 2: 2846, 3: 18851}


## Age-specific contact graph
LAS graph of interchromosomal contacts (nodes = 250 kb loci; edges labelled `Young`, `Old` or `Shared`). Its loci carry
gene-body gene lists; they are replaced with one TSS bin per protein-coding gene
(`../../metadata/hg38/build_hg38_tss_annotation.py`; the same rule as Figures 4–6).

In [4]:
G_all = pickle.load(open(G_ALL, "rb"))
tss_table = pd.read_csv(f"{META}/hg38/bins_protein_coding_genes_tss_hg38_250000.csv", index_col=0)
tss_genes = tss_table["protein_coding_genes_tss"]
BIN_GENES = {i: v.split(",") for i, v in tss_genes.items() if isinstance(v, str) and v}   # bin -> its TSS genes
nx.set_node_attributes(G_all, {n: BIN_GENES.get(n, []) for n in G_all.nodes}, "genes")

edge_type = nx.get_edge_attributes(G_all, "edge_type")
node_chrom = nx.get_node_attributes(G_all, "chrom")
gene_bins = defaultdict(set)                                    # gene -> its TSS bin in the graph
for b, gs in nx.get_node_attributes(G_all, "genes").items():
    for g in gs:
        gene_bins[g].add(b)
bin_chrom = {b: int(c.removeprefix("chr")) for b, c in node_chrom.items()}   # bin -> chromosome number
print(G_all.number_of_nodes(), "loci;", G_all.number_of_edges(), "contacts")

6305 loci; 1557660 contacts


## Pathway colocalization (Figure 1b; also Figure 1c and Supplementary Figure 1Sc)
Per-state hypergeometric test on unique gene–gene contacts (`coloc_stats`), BH over the 60 pathways, plus the Fisher
old-vs-young comparison whose odds ratio (old fold / young fold) is the y-axis of the colocalization-vs-expression
scatters. Writes `data/fig1_pathway_colocalization_tss.csv`.

In [5]:
# genome-wide background: unique gene-gene pairs per contact state
BG = coloc_stats.count_contacts(G_all, "genes")
BG["Total"] = BG["Young"] + BG["Old"] + BG["Shared"]

In [6]:
rows = {}
for pathway, pathway_genes in pathways.items():
    k = coloc_stats.count_contacts(G_all, "genes", pathway_genes)   # the pathway's unique gene-gene pairs per state
    r = {state: k.get(state, 0) for state in ("Young", "Old", "Shared")}
    r["Total"] = sum(r.values())
    for state in ("Young", "Old"):
        r[f"pval_{state}"] = coloc_stats.hypergeom_enrichment_pval(r[state], BG["Total"], BG[state], r["Total"])
        if r["Total"]:
            r[f"fold_{state}"] = (r[state] / r["Total"]) / (BG[state] / BG["Total"])
        else:
            r[f"fold_{state}"] = np.nan
    r["fisher_or"], r["fisher_p"] = coloc_stats.differential_fisher(r["Old"], r["Young"], BG["Old"], BG["Young"])
    rows[pathway] = r

coloc = pd.DataFrame.from_dict(rows, orient="index")
for state in ("Young", "Old"):
    coloc[f"fdr_{state}"] = coloc_stats.bh(coloc[f"pval_{state}"])
coloc["fisher_fdr"] = coloc_stats.bh(coloc.fisher_p)
coloc.to_csv(f"{DATA}/fig1_pathway_colocalization_tss.csv")
print("background unique gene-gene contacts:", BG)
print(f"colocalized: {(coloc.fdr_Young < SIG).sum()} young, {(coloc.fdr_Old < SIG).sum()} old")

background unique gene-gene contacts: {'Old': 731952, 'Shared': 10902642, 'Young': 1102532, 'Total': 12737126}
colocalized: 5 young, 13 old


## Locus-matched permutation test (Supplementary Figure 1Sa)
Each pathway's fold enrichment of age-specific contacts against random gene-bearing loci on the same chromosome, matched
for gene density and expression (quartile of the bin's maximum expression in that age; donor bulk RNA-seq), 10,000 draws,
with the shared `../locus_null.py` (seed = 1000 + pathway position). Writes `data/locus_matched_fibroblast.csv`.

In [7]:
bulk = pd.read_csv(BULK, index_col=0)
bulk = bulk[bulk.gene_type == "protein_coding"].drop_duplicates("gene_name").set_index("gene_name")
expr = {"Young": bulk[["young1", "young2"]].mean(axis=1), "Old": bulk[["old1", "old2"]].mean(axis=1)}

locus_null.prepare(G_all, tss_table, ("Young", "Old"), expr, gene_col="protein_coding_genes_tss")   # node id = TSS-table row
locus = locus_null.test_family(pathways)
locus.to_csv(f"{DATA}/locus_matched_fibroblast.csv", index=False)
print({s: int((locus[f"locus_p_{s}"] < 0.05).sum()) for s in ("Young", "Old")}, "pathways at p < 0.05")

{'Young': 4, 'Old': 6} pathways at p < 0.05


## Pathway expression (GSEA) (Supplementary Figure 1Sb; also Figure 1c)
Preranked GSEA of the Hi-C donors (75 vs 10 yo): protein-coding genes ranked by the DESeq2 Wald statistic, against the
60-pathway panel (shared `../gsea.py`: 1,000 permutations, seed 6). Writes
`data/75_vs_10_yo_gseapy/gseapy.gene_set.prerank.report.csv`. The population-cohort GSEA is in `fibroblast_cohort.ipynb`.

In [8]:
deseq = pd.read_csv(DONOR_DESEQ, index_col=0)
deseq = deseq[deseq["gene_type"] == "protein_coding"]
ranking = gsea.deseq_rank(deseq, genes="gene_name")    # ranked by the DESeq2 Wald statistic
donor_gsea = gsea.prerank(ranking, GMT)
os.makedirs(f"{DATA}/75_vs_10_yo_gseapy", exist_ok=True)
donor_gsea.to_csv(f"{DATA}/75_vs_10_yo_gseapy/gseapy.gene_set.prerank.report.csv", index=False)
print(f"{(donor_gsea['FDR q-val'] < SIG).sum()} of {len(donor_gsea)} pathways differentially expressed (FDR < {SIG})")

2026-10-21 01:44:05,673 [WARNING] Duplicated values found in preranked stats: 10.17% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


12 of 60 pathways differentially expressed (FDR < 0.05)


## Population cohort interferon score (Figure 1d)
Spearman correlation of the per-donor interferon-γ response score with age in the 133 healthy donors of the Fleischer
cohort (scores from `fibroblast_cohort.ipynb` → `data/cohort_isg_scores.csv`). Writes `data/cohort_ifng_age_spearman.csv`.

In [9]:
isg = pd.read_csv(f"{DATA}/cohort_isg_scores.csv", index_col=0)
rho, p = spearmanr(isg.Age, isg["IFN-γ response"])
spearman_table = pd.DataFrame([{"score": "IFN-γ response", "n_donors": len(isg), "spearman_rho": rho, "p": p}])
spearman_table.to_csv(f"{DATA}/cohort_ifng_age_spearman.csv", index=False)
print(f"IFN-γ response vs age: rho = {rho:.3f}, p = {p:.2g}, n = {len(isg)}")

IFN-γ response vs age: rho = 0.262, p = 0.0023, n = 133


## State-specific k-cores (Figure 1e, Supplementary Figures 1Se, 1Sf)
Among a pathway's genes, connect pairs whose contact is state-specific (e.g. old-only) and take a k-core of that graph:
the densest, except for interferon-γ (the 3-core; see below); every gene pair in the core is then labelled with its contact state (majority edge type over the
genes' bins; intrachromosomal pairs have no interchromosomal contact). Shared `../kcore.py`, the same method as the module heatmaps of Figures 4–6.
Writes, per core, `data/kcores/<core>_genes.csv` (genes in genomic order with chromosome) and
`data/kcores/<core>_states.csv` (gene × gene contact-state codes), plus `data/kcores/kcore_summary.csv`.

In [10]:
STATE_CODE = {"Young": 1, "Old": 2, "Shared": 3}                # heatmap codes; 0 = no contact

In [11]:
# (core name, pathway, state, k): IFN-γ uses the 3-core (18 genes, 7 chromosomes) rather than the densest 4-core
# (11 genes on 3 chromosomes); the young-specific cores are the densest
CORES = [("ifng_old_3core", "Interferon gamma response (H)", "Old", 3),
         ("seninh_young_core", "Senescence inhibitors (CellAge)", "Young", None),
         ("ecm_young_core", "Extracellular matrix (Reactome)", "Young", None)]

os.makedirs(f"{DATA}/kcores", exist_ok=True)
summary = []
for name, pathway, state, k in CORES:
    genes = [g for g in pathways[pathway] if g in gene_bins]
    module, k = kcore.state_kcore(edge_type, gene_bins, genes, state, bin_chrom, k=k)
    mat = kcore.state_matrix(edge_type, gene_bins, module, STATE_CODE)
    chroms = [bin_chrom[min(gene_bins[g])] for g in module]
    pd.DataFrame({"gene": module, "chrom": chroms}).to_csv(f"{DATA}/kcores/{name}_genes.csv", index=False)
    pd.DataFrame(mat, index=module, columns=module).to_csv(f"{DATA}/kcores/{name}_states.csv")
    n = {s: int((mat == c).sum() // 2) for s, c in STATE_CODE.items()}
    summary.append({"core": name, "pathway": pathway, "state": state, "k": k, "n_genes_in_graph": len(genes),
                    "n_core_genes": len(module), "n_chromosomes": len(set(chroms)),
                    "pairs_young_only": n["Young"], "pairs_old_only": n["Old"], "pairs_shared": n["Shared"]})
    print(f"{pathway}: {len(genes)} genes in the graph -> {state}-only {k}-core, {len(module)} genes on "
          f"{len(set(chroms))} chromosomes | Young-only {n['Young']}, Old-only {n['Old']}, Shared {n['Shared']}")
pd.DataFrame(summary).to_csv(f"{DATA}/kcores/kcore_summary.csv", index=False)

# for reference: the densest interferon-γ core, which Figure 1e does not show
ifng_genes = [g for g in pathways["Interferon gamma response (H)"] if g in gene_bins]
densest, k_max = kcore.state_kcore(edge_type, gene_bins, ifng_genes, "Old", bin_chrom)
print(f"densest old-only interferon-γ core: {k_max}-core, {len(densest)} genes on "
      f"{len({bin_chrom[min(gene_bins[g])] for g in densest})} chromosomes")

Interferon gamma response (H): 125 genes in the graph -> Old-only 3-core, 18 genes on 7 chromosomes | Young-only 0, Old-only 43, Shared 30
Senescence inhibitors (CellAge): 104 genes in the graph -> Young-only 4-core, 22 genes on 11 chromosomes | Young-only 51, Old-only 0, Shared 52
Extracellular matrix (Reactome): 214 genes in the graph -> Young-only 6-core, 37 genes on 11 chromosomes | Young-only 153, Old-only 15, Shared 133
densest old-only interferon-γ core: 4-core, 11 genes on 3 chromosomes
